# 01 — Unified generator benchmark

Operational specification: [`docs/PROTOCOL.md`, Section 2](../../docs/PROTOCOL.md#2-generator-benchmark).

This validation-only notebook answers RQ1 under a frozen registry and protocol. It audits RAW and
FILTERED artifacts, extracts features with locally verified InceptionV3 and RAD-DINO encoders, and
computes distribution, diversity, duplication, and similarity evidence without opening the test
split. KID/FID use all positive validation references against the canonical 1,361-image synthetic
pool; PRDC and repeated 80% subsamples use balanced, recorded draws. RAW quality defects remain
visible as descriptive warnings, whereas official within-family ranking requires strictly valid
FILTERED pools.

## 1. Load the frozen protocol and configure execution

The first code cell locates the repository, loads the generator protocol and registry, and defines
the real-run and candidate-audit-refresh switches. GPU selection is resolved at
runtime rather than by a workstation-specific UUID, and every output is namespaced under the
canonical benchmark root.

In [ ]:
from pathlib import Path
import csv
import datetime as dt
import numpy as np
import os
import sys
ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'configs').is_dir())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'notebooks/utility'))
from notebooks.utility.generator_benchmark import (
    BENCHMARK_ROOT, CANONICAL_OUTPUTS, FEATURE_SPACES, REPRESENTATIONS, NonFiniteEmbeddingError, atomic_json,
    build_synthetic_duplication_rows, build_train_memorization_rows, candidate_audit_document_rows,
    build_validation_similarity_rows, deterministic_sample, audit_runtime_generator_assets, diversity_metrics, efficiency_from_manifest,
    detect_duplicate_generator_identities, eligibility_failures, evaluation_subset_size, FrozenLocalFeatureExtractor, get_or_extract_embeddings,
    inception_v3_identity, rad_dino_identity,
    load_protocol, load_registry, metadata_positive_paths, plot_generator_summary,
    paired_kid_differences, rank_generator_family, render_similarity_panel, repeated_distribution_metrics,
    representation_preflight_rows, require_official_family_coverage,
    save_resampling_plan, technical_validity_row, training_corpus_from_metadata, write_csv_rows, balanced_subsample_indices,
)
protocol = load_protocol(ROOT)
registry = load_registry(ROOT)
REFRESH_CANDIDATE_AUDIT = True      # refresh metadata/runtime audit before computing metrics
RUN_REAL_BENCHMARK = False           # final execution: open images, load/cache encoders and recompute outputs
BENCHMARK_GPU_SELECTOR = os.environ.get('MAMMODIFFUSION_BENCHMARK_GPU', 'auto')
if RUN_REAL_BENCHMARK:
    # Resolve the largest-memory host GPU at runtime unless an explicit portable selector is supplied.
    from notebooks.utility.classifier_experiment import configure_visible_gpu
    configure_visible_gpu(BENCHMARK_GPU_SELECTOR)
DIVERSITY_PAIR_COUNT = 256
OUTPUT_ROOT = ROOT / BENCHMARK_ROOT
TRUST_EXISTING_LOCAL_DATA = True  # used only when RUN_REAL_BENCHMARK is explicitly enabled
LOCAL_IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

def existing_pool_samples(entry, representation):
    """Load a registered pool only during an explicitly enabled real benchmark."""
    if not RUN_REAL_BENCHMARK:
        raise RuntimeError('pool access is disabled in review-only mode')
    relative_pool = entry['samples'][f'{representation}_positive']
    pool = (ROOT / relative_pool).resolve()
    pool.relative_to(ROOT)
    if 'test' in {part.lower() for part in pool.parts}:
        raise ValueError(f'Test data cannot be used by the generator benchmark: {pool}')
    # Local image files are authoritative; sample IDs are derived deterministically from the pool.
    paths = sorted(path for path in pool.iterdir() if path.is_file() and path.suffix.lower() in LOCAL_IMAGE_SUFFIXES)
    ids = [f"{entry['id']}::{representation}::{path.name}" for path in paths]
    return paths, ids

{'refresh_candidate_audit': REFRESH_CANDIDATE_AUDIT, 'run_real_benchmark': RUN_REAL_BENCHMARK, 'protocol': protocol, 'expected_outputs': CANONICAL_OUTPUTS}

## 2. Discover candidates and verify declared identities

Every benchmark-enabled registry entry is inspected against its checkpoint, sampling configuration,
RAW/FILTERED directories, filter report, and VAE/base-model dependencies.
The local Run All treats the readable registered files as authoritative execution inputs. Candidate roles remain explicit so an ablation or descriptive
baseline cannot be mistaken for an independent primary model.

In [ ]:
candidate_audits = []
if RUN_REAL_BENCHMARK:
    candidate_audits = detect_duplicate_generator_identities([
        audit_runtime_generator_assets(ROOT, entry, protocol)
        for entry in registry['generators']
        if entry.get('benchmark', {}).get('enabled', False)
    ])
    if TRUST_EXISTING_LOCAL_DATA:
        entries_by_id = {entry['id']: entry for entry in registry['generators']}
        for audit in candidate_audits:
            entry = entries_by_id[audit['generator_id']]
            concrete_ready = True
            for representation in REPRESENTATIONS:
                paths, _ = existing_pool_samples(entry, representation)
                audit['representations'][representation]['count'] = len(paths)
                concrete_ready &= len(paths) >= protocol['synthetic_pool_target']
            audit['eligible_for_benchmark_execution'] = bool(concrete_ready)
            audit['eligible_for_descriptive_benchmark'] = bool(concrete_ready)
            audit['distinct_generator_for_ranking'] = bool(
                concrete_ready and entry.get('distinct_generator_for_ranking', False)
            )
            audit['eligible_for_official_family_ranking'] = bool(
                audit['distinct_generator_for_ranking']
                and entry.get('eligible_for_downstream_selection', False)
                and entry.get('candidate_role', 'primary_candidate') == 'primary_candidate'
            )
            audit['execution_data_policy'] = 'current_local_files'
candidate_audits if RUN_REAL_BENCHMARK else {'status': 'review-only; data and experiments were not inspected'}

## 3. Apply execution and downstream-selection eligibility rules

Descriptive execution requires each registered RAW and FILTERED directory to contain at least the
protocol target of readable local images. Historical fingerprint mismatches are retained as
diagnostics but do not suppress computation. Candidate role, declared downstream eligibility,
FILTERED technical validity, and the active ranking gates still determine whether a row can enter
an official family ranking; ablations and descriptive baselines remain non-selectable.

In [ ]:
eligible = [row for row in candidate_audits if row['eligible_for_benchmark_execution']]
if RUN_REAL_BENCHMARK and not eligible:
    raise RuntimeError('No registered candidate has complete local RAW and FILTERED image pools.')
[(row['generator_id'], row['candidate_role'], row['eligible_for_downstream_selection'], row['blockers']) for row in candidate_audits]

## 4. Audit registered RAW and FILTERED pool counts

Counts are derived from the current registered directories. Recorded sample IDs are preserved when
their CSV is readable; otherwise deterministic IDs are constructed from the current top-level files.
This audit verifies that every representation contains at least 1,361 usable images. The later metric
stage deterministically selects exactly 1,361 images from larger pools, so discovered RAW counts of
2,722 or 4,083 are valid and are not misreported as exact 1,361-image directories.

In [ ]:
counts = [{
    'generator_id': row['generator_id'],
    **{name: row['representations'][name]['count'] for name in REPRESENTATIONS}
} for row in candidate_audits]
counts

## 5. Construct the protected real reference set

Positive validation references are resolved from `data/processed/metadata/val.csv`, with canonical
sample identifiers retained for deterministic sampling and cache validation. The code explicitly
rejects test paths and records the reference count. Training images are not used for distribution
quality metrics; they are loaded later only for the separate memorization analysis.

In [ ]:
validation_paths = validation_ids = None
if RUN_REAL_BENCHMARK:
    required_metadata = [ROOT / 'data/processed/metadata/val.csv']
    missing = [str(path) for path in required_metadata if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Required benchmark metadata is missing: {missing}')
    validation_paths, validation_ids = metadata_positive_paths(ROOT, 'data/processed/metadata/val.csv')
    real_reference_count = len(validation_paths)
    reference_status = {'real_reference_count': real_reference_count, 'memorization_policy': 'processed train and traditional augmentation metadata'}
else:
    reference_status = {'status': 'Deferred: no real dataset was opened', 'policy': 'validation positives for quality; existing train and augmentation metadata for memorization'}
reference_status

## 6. Separate feature extractability from image-quality validity

Each declared image is checked for readability, shape, finite range, near-black content, and exact
duplication under representation-aware rules. RAW defects are reported while still allowing
descriptive extraction when technically possible; FILTERED defects are fatal to official ranking.
Audit tables and family-preflight counts are written independently of the expensive metric run so
metadata-only review remains possible.

In [ ]:
candidate_audit_rows = candidate_audit_document_rows(candidate_audits)
if RUN_REAL_BENCHMARK and REFRESH_CANDIDATE_AUDIT:
    write_csv_rows(OUTPUT_ROOT / 'candidate_audit.csv', candidate_audit_rows)
technical_rows = candidate_preflight = family_preflight_counts = None
if RUN_REAL_BENCHMARK:
    technical_rows = list()
    for audit in candidate_audits:
        if not audit['eligible_for_descriptive_benchmark']:
            continue
        entry = next(item for item in registry['generators'] if item['id'] == audit['generator_id'])
        for representation in REPRESENTATIONS:
            paths, _ = existing_pool_samples(entry, representation)
            technical_rows.append(technical_validity_row(audit['generator_id'], representation, paths, minimum_unique=protocol['synthetic_pool_target'], maximum_exact_duplicate_rate=protocol['eligibility_gates']['maximum_exact_duplicate_rate']))
    write_csv_rows(OUTPUT_ROOT / 'technical_validity.csv', technical_rows)
    candidate_preflight = representation_preflight_rows(candidate_audits, technical_rows)
    family_preflight_counts = require_official_family_coverage(candidate_preflight)
{'technical_validity': technical_rows, 'candidate_preflight': candidate_preflight, 'official_filtered_by_family': family_preflight_counts} if technical_rows is not None else 'Deferred until RUN_REAL_BENCHMARK=True'

## 7. Verify frozen local encoders and content-aware caches

The benchmark resolves local InceptionV3 weights and the complete RAD-DINO snapshot through portable
paths, hashes weights and preprocessing identities, and disables model downloads. Each frozen encoder
is loaded once, checked on a deterministic preflight batch, and reused across cache misses. Embedding
caches are accepted only when sample IDs, current image fingerprints, code version, model identity,
and preprocessing metadata all match; otherwise features are recomputed and the invalidation reason
is recorded.

In [ ]:
embedding_cache_root = OUTPUT_ROOT / 'embedding_cache'
_torch_cache_root = Path(os.environ.get('TORCH_HOME', Path(os.environ.get('XDG_CACHE_HOME', Path.home() / '.cache')) / 'torch')).expanduser()
_inception_default = _torch_cache_root / 'hub/checkpoints/inception_v3_google-0cc3c7bd.pth'
INCEPTION_CHECKPOINT_PATH = Path(os.environ.get('MAMMODIFFUSION_INCEPTION_CHECKPOINT', _inception_default)).expanduser()
RAD_DINO_SNAPSHOT_PATH = Path(os.environ.get('MAMMODIFFUSION_RAD_DINO_SNAPSHOT', ROOT / 'notebooks/pretrained_model/rad-dino')).expanduser()
ENCODER_IDENTITIES = {}
candidate_features = reference_features = candidate_paths = candidate_ids = None
feature_extraction_failures = cache_events = feature_family_counts = None
if RUN_REAL_BENCHMARK:
    if INCEPTION_CHECKPOINT_PATH is None or RAD_DINO_SNAPSHOT_PATH is None:
        raise RuntimeError('Set both local encoder paths; the benchmark will not download models or use generic weight identifiers.')
    import torch
    import torchvision
    inception_path, rad_path = Path(INCEPTION_CHECKPOINT_PATH), Path(RAD_DINO_SNAPSHOT_PATH)
    LOCAL_ENCODER_PATHS = {'inception_v3': inception_path, 'rad_dino': rad_path}
    _local_encoder_extractors = {}
    def local_feature_extractor(feature_space):  # one frozen model per space, reused across cache misses
        extractor = _local_encoder_extractors.get(feature_space)
        if extractor is None:
            extractor = FrozenLocalFeatureExtractor(feature_space, LOCAL_ENCODER_PATHS[feature_space], device='cuda:0')
            _local_encoder_extractors[feature_space] = extractor
        return extractor
    def extract_local_features(paths, feature_space):
        return local_feature_extractor(feature_space).extract(paths)
    def close_local_encoders():
        while _local_encoder_extractors:
            _, extractor = _local_encoder_extractors.popitem()
            extractor.close()
    ENCODER_IDENTITIES['inception_v3'] = inception_v3_identity(torchvision.__version__, 'Inception_V3_Weights.IMAGENET1K_V1', inception_path, {'resize': 299, 'normalization': 'weights-enum-default'})
    ENCODER_IDENTITIES['rad_dino'] = rad_dino_identity('microsoft/rad-dino', rad_path, commit_hash=rad_path.name if rad_path.parent.name == 'snapshots' else None, config_path=rad_path / 'config.json', weight_paths=sorted(rad_path.glob('*.safetensors')), processor_config_path=rad_path / 'preprocessor_config.json', preprocessing_configuration={'processor': 'local AutoImageProcessor configuration'})
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is required for the real benchmark')
    gpu_properties = torch.cuda.get_device_properties(0); observed_gpu_uuid = str(gpu_properties.uuid)
    requested_gpu_uuid = os.environ.get('CUDA_VISIBLE_DEVICES', '')
    if requested_gpu_uuid.startswith('GPU-') and observed_gpu_uuid.lower() != requested_gpu_uuid[4:].lower():
        raise RuntimeError(f'Requested GPU UUID {requested_gpu_uuid}, but CUDA local 0 reports {observed_gpu_uuid}')
    extractor_preflight = {}
    preflight_paths = validation_paths[:min(2, len(validation_paths))]
    if not preflight_paths:
        raise RuntimeError('Extractor preflight requires at least one validation image')
    for extractor_name in FEATURE_SPACES:
        torch.cuda.reset_peak_memory_stats(0)
        first = extract_local_features(preflight_paths, extractor_name)
        second = extract_local_features(preflight_paths, extractor_name)
        expected_dimension = 2048 if extractor_name == 'inception_v3' else 768
        if first.shape != (len(preflight_paths), expected_dimension) or not np.isfinite(first).all():
            raise RuntimeError(f'{extractor_name} preflight returned invalid features: {first.shape}')
        if not np.array_equal(first, second):
            raise RuntimeError(f'{extractor_name} preflight is not deterministic')
        extractor_preflight[extractor_name] = {'status': 'pass', 'sample_count': len(preflight_paths), 'feature_shape': list(first.shape), 'finite': True, 'deterministic': True, 'evaluation_mode': True, 'gradients_enabled': False, 'peak_memory_bytes': int(torch.cuda.max_memory_allocated(0))}
        del first, second
        torch.cuda.empty_cache()
    execution_config = {'schema_version': 1, 'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(), 'status': 'running', 'repository_root': str(ROOT), 'working_directory': str(Path.cwd()), 'protocol': 'configs/generator_benchmark_protocol.json', 'run_real_benchmark': True, 'python_version': sys.version, 'pytorch_version': torch.__version__, 'torchvision_version': torchvision.__version__, 'cuda_build_version': torch.version.cuda, 'gpu': {'cuda_visible_devices': requested_gpu_uuid, 'local_cuda_index': 0, 'name': torch.cuda.get_device_name(0), 'uuid': observed_gpu_uuid, 'total_memory_bytes': int(gpu_properties.total_memory), 'free_memory_bytes_at_start': int(torch.cuda.mem_get_info(0)[0])}, 'extractor_identities': ENCODER_IDENTITIES, 'extractor_preflight': extractor_preflight, 'stability_repetitions': protocol['resampling']['stability_repetitions'], 'resampling_fraction': protocol['resampling']['subsampling_fraction'], 'nearest_neighbour_k': protocol['resampling']['nearest_neighbour_k'], 'practical_equivalence_margin': protocol['selection']['practical_equivalence_margin'], 'candidates_included': [row['generator_id'] for row in candidate_preflight if row['raw_descriptive_ready'] or row['filtered_descriptive_ready']], 'candidates_excluded': [{'generator_id': row['generator_id'], 'reason': row['block_reasons']} for row in candidate_preflight if not (row['raw_descriptive_ready'] or row['filtered_descriptive_ready'])]}
    atomic_json(OUTPUT_ROOT / 'execution_config.json', execution_config)
    candidate_features, reference_features, candidate_paths, candidate_ids = dict(), dict(), dict(), dict()
    feature_extraction_failures, cache_events = list(), list()
    execution_ids = {row['generator_id'] for row in candidate_audits if row['eligible_for_benchmark_execution']}
    benchmark_keys = {(row['generator_id'], row['condition'].lower()) for row in technical_rows if row['eligible_for_distribution_metrics'] and row['generator_id'] in execution_ids}
    reference_sets = {'validation': (validation_paths, validation_ids, 'data/processed/metadata/val.csv')}
    for pool_name, (paths, ids, manifest) in reference_sets.items():
        for extractor_name in FEATURE_SPACES:
            cache = embedding_cache_root / '_references' / pool_name / f'{extractor_name}.npy'
            reference_features[(pool_name, extractor_name)], cache_metadata = get_or_extract_embeddings(
                cache, paths, ids, extractor=extractor_name, preprocessing='registered frozen extractor preprocessing',
                code_version='embedding-integrity-v3', source_manifest=str(ROOT / manifest), metadata_csv=str(ROOT / manifest),
                extractor_model_id=ENCODER_IDENTITIES[extractor_name].get('model_repository', ENCODER_IDENTITIES[extractor_name].get('weights_enum')), extractor_weights_identifier=ENCODER_IDENTITIES[extractor_name]['identity_sha256'], extractor_identity=ENCODER_IDENTITIES[extractor_name],
                extract_fn=extract_local_features)
            cache_events.append({'scope': 'reference', 'pool': pool_name, 'extractor': extractor_name, 'cache_event': cache_metadata['cache_event'], 'reason': cache_metadata['cache_invalidation_reason']})
    for audit in candidate_audits:
        entry = next(item for item in registry['generators'] if item['id'] == audit['generator_id'])
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            discovered_paths, discovered_ids = existing_pool_samples(entry, representation)
            selected = set(deterministic_sample(discovered_paths, protocol['synthetic_pool_target'], protocol['sampling']['seed']))
            pairs = [(path, sample_id) for path, sample_id in zip(discovered_paths, discovered_ids) if str(path) in selected]
            paths, ids = [item[0] for item in pairs], [item[1] for item in pairs]; candidate_paths[key] = paths; candidate_ids[key] = ids
            for extractor_name in FEATURE_SPACES:
                cache = embedding_cache_root / audit['generator_id'] / representation / f'{extractor_name}.npy'
                try:
                    candidate_features[(audit['generator_id'], representation, extractor_name)], cache_metadata = get_or_extract_embeddings(
                        cache, paths, ids, extractor=extractor_name, preprocessing='registered frozen extractor preprocessing',
                        code_version='embedding-integrity-v3',
                        extractor_model_id=ENCODER_IDENTITIES[extractor_name].get('model_repository', ENCODER_IDENTITIES[extractor_name].get('weights_enum')), extractor_weights_identifier=ENCODER_IDENTITIES[extractor_name]['identity_sha256'], extractor_identity=ENCODER_IDENTITIES[extractor_name],
                        extract_fn=extract_local_features)
                    cache_events.append({'scope': 'candidate', 'generator_id': audit['generator_id'], 'representation': representation, 'extractor': extractor_name, 'cache_event': cache_metadata['cache_event'], 'reason': cache_metadata['cache_invalidation_reason']})
                except NonFiniteEmbeddingError as exc:
                    feature_extraction_failures.extend({'generator_id': audit['generator_id'], 'representation': representation, **failure} for failure in exc.failures)
                    benchmark_keys.discard(key)
                    for feature_space in FEATURE_SPACES:
                        candidate_features.pop((audit['generator_id'], representation, feature_space), None)
                    break
    feature_preflight = [{**row, 'filtered_official_ranking_ready': row['filtered_official_ranking_ready'] and all((row['generator_id'], 'filtered', extractor) in candidate_features for extractor in FEATURE_SPACES)} for row in candidate_preflight]
    feature_family_counts = require_official_family_coverage(feature_preflight)
feature_status = 'Deferred' if candidate_features is None else {'candidate_feature_sets': len(candidate_features), 'reference_feature_sets': len(reference_features), 'cache_events': cache_events, 'feature_extraction_failures': feature_extraction_failures, 'official_filtered_by_family': feature_family_counts}
feature_status

## 8. Compute distribution fidelity and coverage metrics

For each technically eligible representation, full-pool KID/FID estimates are computed in both
feature spaces, while PRDC uses balanced real/synthetic subsets and the registered neighborhood size.
Metric inputs come from the same verified embeddings and every deterministic sample plan is retained.
Inception and RAD-DINO quantify different representations of similarity, and neither should be read
as a clinical-validity score.

In [ ]:
distribution_summaries = distribution_repetitions = None
if RUN_REAL_BENCHMARK:
    distribution_summaries, distribution_repetitions = list(), list()
    stability_size = evaluation_subset_size(protocol['synthetic_pool_target'], len(validation_ids), protocol['synthetic_pool_target'], protocol['resampling']['subsampling_fraction'])
    shared_resampling_plan = balanced_subsample_indices(len(validation_ids), protocol['synthetic_pool_target'], stability_size, protocol['resampling']['stability_repetitions'], protocol['sampling']['seed'], nearest_neighbour_k=protocol['resampling']['nearest_neighbour_k'])
    save_resampling_plan(OUTPUT_ROOT / 'resampling_plan.json', shared_resampling_plan, protocol)
    eligibility = {(row['generator_id'], row['condition'].lower()): row['eligible_for_distribution_metrics'] for row in technical_rows}
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            if not eligibility.get((audit['generator_id'], representation), False):  # excluded generators (e.g. G06) are absent from technical_rows
                continue
            for extractor_name in FEATURE_SPACES:
                records, summary = repeated_distribution_metrics(reference_features[('validation', extractor_name)], candidate_features[(audit['generator_id'], representation, extractor_name)], protocol, resampling_plan=shared_resampling_plan)
                distribution_repetitions.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), 'extractor': extractor_name, **record, 'real_ids': [validation_ids[index] for index in record['real_indices']], 'synthetic_ids': [candidate_ids[(audit['generator_id'], representation)][index] for index in record['synthetic_indices']]} for record in records)
                flat = {**summary['full_pool_distribution_estimates'], **summary['balanced_prdc_point_estimates'], **{f'{metric}_stability_{field}': value for metric, values in summary['stability_estimates'].items() for field, value in values.items()}}
                distribution_summaries.append({'generator_id': audit['generator_id'], 'condition': representation.upper(), 'extractor': extractor_name, **flat, 'full_pool_real_count': summary['full_pool_real_count'], 'full_pool_synthetic_count': summary['full_pool_synthetic_count'], 'balanced_prdc_point_real_count': summary['balanced_prdc_point_real_count'], 'balanced_prdc_point_synthetic_count': summary['balanced_prdc_point_synthetic_count'], 'stability_subset_size': summary['stability_subset_size'], 'stability_interval_type': summary['stability_interval_type'], 'full_pool_distribution_policy': summary['full_pool_distribution_policy'], 'fid_full_pool_caveat': summary['fid_full_pool_caveat']})
    write_csv_rows(OUTPUT_ROOT / 'distribution_metrics_repetitions.csv', distribution_repetitions)
    write_csv_rows(OUTPUT_ROOT / 'distribution_metrics_summary.csv', distribution_summaries)
distribution_summaries if distribution_summaries is not None else {'status': 'Deferred', 'resampling': protocol['resampling']}

## 9. Quantify within-pool diversity

The diversity analysis draws a fixed number of deterministic image pairs and evaluates MS-SSIM-based
similarity for each candidate representation. Pair indices are recorded so reruns use the same
comparison plan. The result characterizes redundancy within a generated pool but does not by itself
measure fidelity to real mammography.

In [ ]:
diversity_rows = None
if RUN_REAL_BENCHMARK:
    diversity_rows = list()
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            diversity_rows.append({'generator_id': audit['generator_id'], 'condition': representation.upper(),
                                   **diversity_metrics(candidate_paths[key], candidate_features[(*key, 'rad_dino')], pair_count=DIVERSITY_PAIR_COUNT, seed=protocol['sampling']['seed'])})
    write_csv_rows(OUTPUT_ROOT / 'diversity_metrics.csv', diversity_rows)
diversity_rows if diversity_rows is not None else 'Deferred; mandatory metrics are synthetic NN distance, perceptual-hash duplicate rate and deterministic MS-SSIM diversity. LPIPS is optional.'

## 10. Audit exact, perceptual, and manually confirmed duplication

The current registered synthetic samples are examined for byte/content duplicates and perceptual-hash
neighborhoods;
the workflow keeps exact, pHash-only, and manually confirmed rates as distinct quantities. This
prevents a heuristic near-duplicate flag from being reported as a confirmed copy. The resulting
evidence feeds the technical safety gates defined by the active protocol.

In [ ]:
synthetic_duplication_rows = None
if RUN_REAL_BENCHMARK:
    synthetic_duplication_rows = list()
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            path_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            rows = build_synthetic_duplication_rows(candidate_features[(*key, 'rad_dino')], candidate_ids[key], path_map, protocol['memorization']['flag_rule'])
            synthetic_duplication_rows.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), **row} for row in rows)
    write_csv_rows(OUTPUT_ROOT / 'synthetic_duplication.csv', synthetic_duplication_rows)
synthetic_duplication_rows[:5] if synthetic_duplication_rows is not None else 'Deferred'

## 11. Test memorization against the complete training metadata

The pool is built in memory from every real train sample and traditional positive augmentation in
the existing metadata. One RAD-DINO embedding cache is shared for this common corpus, and similarity
thresholds are applied
without reducing the reference to positive images alone. This is the designated train-memorization
analysis; validation similarity remains a separate descriptive question.

In [ ]:
train_memorization_rows = None
if RUN_REAL_BENCHMARK:
    train_memorization_rows = list()
    train_paths, train_ids, train_labels, train_sources = training_corpus_from_metadata(ROOT)
    train_path_map = dict(zip(train_ids, map(Path, train_paths)))
    train_metadata = ROOT / 'data/processed/metadata/train.csv'
    shared_training_cache = embedding_cache_root / 'shared_training_corpus/rad_dino.npy'
    train_features, train_cache_metadata = get_or_extract_embeddings(shared_training_cache, train_paths, train_ids, extractor='rad_dino', preprocessing='local recorded RAD-DINO processor', code_version='embedding-integrity-v4', source_manifest=str(train_metadata), metadata_csv=str(train_metadata), extractor_model_id=ENCODER_IDENTITIES['rad_dino']['model_repository'], extractor_weights_identifier=ENCODER_IDENTITIES['rad_dino']['identity_sha256'], extractor_identity=ENCODER_IDENTITIES['rad_dino'], extract_fn=extract_local_features)
    cache_events.append({'scope': 'training_corpus', 'extractor': 'rad_dino', 'cache_event': train_cache_metadata['cache_event'], 'reason': train_cache_metadata['cache_invalidation_reason']})
    for audit in candidate_audits:
        if not audit['eligible_for_benchmark_execution']:
            continue
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            synthetic_path_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            rows = build_train_memorization_rows(candidate_features[(*key, 'rad_dino')], train_features, candidate_ids[key], train_ids, synthetic_path_map, train_path_map, protocol['memorization']['flag_rule'], train_labels, train_sources)
            train_memorization_rows.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), **row} for row in rows)
    write_csv_rows(OUTPUT_ROOT / 'train_memorization.csv', train_memorization_rows)
    close_local_encoders()
train_memorization_rows[:5] if train_memorization_rows is not None else 'Deferred; only this table controls the memorization gate.'

## 12. Measure validation similarity without conflating it with memorization

Synthetic images are compared with the positive validation reference to identify unusually similar
examples and to render auditable nearest-neighbor evidence. Because validation images were not used
for training, these results describe distributional or sample-level resemblance rather than training
memorization. Paths, sample IDs, scores, and thresholds are retained for traceability.

In [ ]:
validation_similarity_rows = None
if RUN_REAL_BENCHMARK:
    validation_similarity_rows = list(); validation_path_map = dict(zip(validation_ids, map(Path, validation_paths)))
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            synthetic_path_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            rows = build_validation_similarity_rows(candidate_features[(*key, 'rad_dino')], reference_features[('validation', 'rad_dino')], candidate_ids[key], validation_ids, synthetic_path_map, validation_path_map)
            validation_similarity_rows.extend({'generator_id': audit['generator_id'], 'condition': representation.upper(), **row} for row in rows)
    write_csv_rows(OUTPUT_ROOT / 'validation_similarity.csv', validation_similarity_rows)
validation_similarity_rows[:5] if validation_similarity_rows is not None else 'Deferred; this descriptive table never contains a memorization flag.'

## 13. Estimate stability by repeated balanced subsampling

The code repeats the preregistered 80% balanced subsampling plan with paired seeds across candidates
and records every selected index. It summarizes interval bounds, variability, gate-pass fractions,
and paired KID differences while preserving the full-pool point estimates separately. This is
repeated subsampling, not a nonparametric bootstrap, and its intervals describe sensitivity to the
registered sampling plan.

In [ ]:
if RUN_REAL_BENCHMARK:
    example_sizes = {row['generator_id']: evaluation_subset_size(row['representations']['filtered']['count'], real_reference_count, protocol['synthetic_pool_target']) for row in eligible}
    subsampling_status = {'evaluation_subset_size_by_generator': example_sizes, 'sampling': 'balanced without replacement', 'recorded_fields': ['repetition', 'seed', 'real_indices', 'synthetic_indices']}
else:
    subsampling_status = {'status': 'Deferred', 'sampling': 'balanced without replacement', 'seed': protocol['sampling']['seed']}
subsampling_status

## 14. Join evidence, apply gates, and build within-family rankings

Distribution, diversity, duplication, memorization, technical-validity, and
efficiency fields are merged into canonical summary rows. The effective execution fields state the
`trusted_existing_local_files` policy used by this Run All. Candidate role and technical/quality
evidence still control official ranking, and deterministic within-family order follows the registered
hierarchy led by RAD-DINO KID. Both the flat summary and ranking tables are persisted for the selection
notebook.

In [ ]:
generator_summary = generator_ranking = None
if RUN_REAL_BENCHMARK:
    generator_summary = list(); audits_by_id = {row['generator_id']: row for row in candidate_audits}
    distributions = {(row['generator_id'], row['condition'], row['extractor']): row for row in distribution_summaries}
    diversities = {(row['generator_id'], row['condition']): row for row in diversity_rows}
    technical_by_condition = {(row['generator_id'], row['condition']): row for row in technical_rows}
    for technical in technical_rows:
        generator_id, condition = technical['generator_id'], technical['condition']; audit = audits_by_id[generator_id]
        raw_technical = technical_by_condition[(generator_id, 'RAW')]; filtered_technical = technical_by_condition[(generator_id, 'FILTERED')]
        rad = distributions.get((generator_id, condition, 'rad_dino')); inception = distributions.get((generator_id, condition, 'inception_v3')); diversity = diversities.get((generator_id, condition), dict())
        train_group = [row for row in train_memorization_rows if row['generator_id'] == generator_id and row['condition'] == condition]
        validation_group = [row for row in validation_similarity_rows if row['generator_id'] == generator_id and row['condition'] == condition]
        duplication_group = [row for row in synthetic_duplication_rows if row['generator_id'] == generator_id and row['condition'] == condition]
        train_rate = sum(row['memorization_flag'] for row in train_group) / len(train_group) if train_group else None
        duplicate_rate = sum(row['duplicate_flag'] for row in duplication_group) / len(duplication_group) if duplication_group else None
        entry = next(item for item in registry['generators'] if item['id'] == generator_id)
        summary = {'generator_id': generator_id, 'condition': condition, 'family': audit['scientific_family'], 'role': audit['candidate_role'],
                   'eligible_for_selection': audit['eligible_for_downstream_selection'], 'technical_validity': technical['eligible_for_distribution_metrics'],
                   'technical_validity_rate': technical['technical_validity_rate'], 'feature_extractable_rate': technical['feature_extractable_rate'], 'quality_validity_rate': technical['quality_validity_rate'],
                   'raw_feature_extractable_rate': raw_technical['feature_extractable_rate'], 'raw_quality_validity_rate': raw_technical['quality_validity_rate'], 'raw_near_black_rate': raw_technical['n_near_black'] / raw_technical['n_discovered'], 'raw_constant_range_rate': raw_technical['n_invalid_range'] / raw_technical['n_discovered'],
                   'filtered_feature_extractable_rate': filtered_technical['feature_extractable_rate'], 'filtered_quality_validity_rate': filtered_technical['quality_validity_rate'],
                   'eligible_for_official_ranking': condition == 'FILTERED' and technical['eligible_for_official_ranking'] and audit['eligible_for_official_family_ranking'],
                   'filter_acceptance_rate': audit.get('filter_acceptance_rate'), 'valid_positive_images': technical['n_unique_quality_valid_content'],
                   'n_corrupt': technical['n_corrupt'],
                   'raddino_kid': rad.get('kid_full_pool') if rad else None, 'raddino_kid_stability_low': rad.get('kid_stability_percentile_2_5') if rad else None,
                   'raddino_kid_stability_high': rad.get('kid_stability_percentile_97_5') if rad else None, 'raddino_kid_std': rad.get('kid_stability_standard_deviation') if rad else None,
                   'raddino_precision': rad.get('precision_balanced_point') if rad else None, 'raddino_recall': rad.get('recall_balanced_point') if rad else None,
                   'raddino_density': rad.get('density_balanced_point') if rad else None, 'raddino_coverage': rad.get('coverage_balanced_point') if rad else None,
                   'raddino_fid': rad.get('fid_full_pool') if rad else None, 'inception_kid': inception.get('kid_full_pool') if inception else None,
                   'inception_fid': inception.get('fid_full_pool') if inception else None, 'stability_interval_type': rad.get('stability_interval_type') if rad else None, 'ms_ssim_diversity': diversity.get('ms_ssim_diversity'),
                   'synthetic_duplicate_rate': duplicate_rate, 'synthetic_exact_duplicate_rate': diversity.get('synthetic_exact_duplicate_rate'),
                   'perceptual_hash_duplicate_rate': diversity.get('perceptual_hash_duplicate_rate'), 'train_memorization_rate': train_rate,
                   'validation_nearest_neighbour_distance': float(np.mean([row['embedding_distance'] for row in validation_group])) if validation_group else None,
                   **efficiency_from_manifest(ROOT, entry), 'metrics_complete': rad is not None and inception is not None, 'test_access': False}
        summary['technical_gates_passed'] = technical['eligible_for_distribution_metrics'] and not eligibility_failures(summary, protocol['eligibility_gates'])
        generator_summary.append(summary)
    write_csv_rows(OUTPUT_ROOT / 'generator_summary.csv', generator_summary)
    generator_ranking = list()
    filtered_rows = [row for row in generator_summary if row['condition'] == 'FILTERED' and row['eligible_for_official_ranking']]
    for family in ('finetuned', 'from_scratch'):
        generator_ranking.extend(rank_generator_family(filtered_rows, family, protocol['eligibility_gates']))
    write_csv_rows(OUTPUT_ROOT / 'generator_ranking.csv', generator_ranking)
    paired_rows = list()
    for family in ('finetuned', 'from_scratch'):
        top = [row for row in generator_ranking if row['family'] == family and row['eligible']][:2]
        if len(top) == 2:
            left = [row for row in distribution_repetitions if row['generator_id'] == top[0]['generator_id'] and row['condition'] == 'FILTERED' and row['extractor'] == 'rad_dino']
            right = [row for row in distribution_repetitions if row['generator_id'] == top[1]['generator_id'] and row['condition'] == 'FILTERED' and row['extractor'] == 'rad_dino']
            paired_rows.append({'family': family, **{key: value for key, value in paired_kid_differences(left, right, top[0]['generator_id'], top[1]['generator_id']).items() if key != 'paired_differences'}})
    if paired_rows: write_csv_rows(OUTPUT_ROOT / 'paired_generator_differences.csv', paired_rows)
    figure = plot_generator_summary(generator_summary); (OUTPUT_ROOT / 'figures').mkdir(parents=True, exist_ok=True); figure.savefig(OUTPUT_ROOT / 'figures/generator_summary.png', dpi=150)
if not RUN_REAL_BENCHMARK:
    def _saved_rows(name):
        path = OUTPUT_ROOT / name
        if not path.is_file():
            return None
        with path.open(newline='', encoding='utf-8') as stream:
            return list(csv.DictReader(stream))
    generator_summary = _saved_rows('generator_summary.csv')
    generator_ranking = _saved_rows('generator_ranking.csv')
generator_summary if generator_summary is not None else {'status': 'small saved summary is missing; run the real benchmark explicitly'}

## 15. Present the deterministic ranking and Pareto view

The ranking view reports eligible and descriptive candidates under the exact tie-break order encoded
by the protocol. The accompanying Pareto analysis visualizes competing fidelity, coverage, diversity,
and efficiency dimensions without creating a new decision rule. It is therefore an interpretive aid,
not an alternative post hoc optimizer.

In [ ]:
ranking_view = generator_ranking if generator_ranking is not None else 'Deferred; ranking order is eligibility → RAD-DINO KID → coverage → precision → RAD-DINO FID → Inception KID → stability → generator_id.'
ranking_view

## 16. Render deterministic qualitative audit panels

For each candidate, seeded image panels and nearest-neighbor panels are generated from the same
benchmark sample IDs and current local paths used by the quantitative analysis. Captions preserve
generator, representation, synthetic-sample, and reference identities. These panels support human
inspection of artifacts and suspicious similarities but do not override the registered quantitative
gates or ranking.

In [ ]:
panel_outputs = None
if RUN_REAL_BENCHMARK:
    panel_outputs = list(); panel_root = OUTPUT_ROOT / 'diagnostic_panels'
    for audit in candidate_audits:
        for representation in REPRESENTATIONS:
            key = (audit['generator_id'], representation)
            if key not in benchmark_keys:
                continue
            condition = representation.upper(); synthetic_map = dict(zip(candidate_ids[key], candidate_paths[key]))
            train_panel = [{**row, 'source_id': row['nearest_train_id']} for row in train_memorization_rows if row['generator_id'] == audit['generator_id'] and row['condition'] == condition]
            validation_panel = [{**row, 'source_id': row['nearest_validation_id']} for row in validation_similarity_rows if row['generator_id'] == audit['generator_id'] and row['condition'] == condition]
            synthetic_panel = [{**row, 'source_id': row['nearest_synthetic_id']} for row in synthetic_duplication_rows if row['generator_id'] == audit['generator_id'] and row['condition'] == condition]
            for name, rows, references in (('train', train_panel, train_path_map), ('validation', validation_panel, validation_path_map), ('synthetic', synthetic_panel, synthetic_map)):
                output = panel_root / f"{audit['generator_id']}_{representation}_{name}.png"
                panel_outputs.append(str(render_similarity_panel(rows, synthetic_map, references, output, f"{audit['generator_id']} {condition}: synthetic | nearest {name}")))
    execution_config.update({'status': 'complete', 'completed_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(), 'cache_hits': sum(row['cache_event'] == 'hit' for row in cache_events), 'cache_misses': sum(row['cache_event'] == 'miss' for row in cache_events), 'cache_events': cache_events, 'feature_extraction_failures': feature_extraction_failures})
    atomic_json(OUTPUT_ROOT / 'execution_config.json', execution_config)
panel_outputs if panel_outputs is not None else {'status': 'Deferred', 'policy': 'deterministic closest / median / farthest for train, validation and synthetic'}